In [ ]:
import pandas as pd
import geopandas as gpd
import os, glob
import matplotlib.pyplot as plt
import cartopy.crs as ccrs
import cartopy.feature as cfeature
from cartopy.mpl.gridliner import LONGITUDE_FORMATTER, LATITUDE_FORMATTER
import matplotlib.ticker as mticker
import numpy as np
import seaborn as sns
import matplotlib.cm as cm
import matplotlib.colors as mcolors

In [ ]:
from matplotlib import font_manager
# Manually register the font file with matplotlib's fontManager.
font_manager.fontManager.addfont('../../data/Arial.ttf')

# Verify the font was registered.
print([f.name for f in font_manager.fontManager.ttflist if 'Arial' in f.name])
plt.rcParams['font.family'] = 'Arial'

In [ ]:
base_folder = '../../data/inter'
countries = [
    'Australia', 'Brazil', 'China',  'US', 'France', 'Portugal', 'Nigeria'
    # 'DHS_new',
]
files = []
for country in countries:
    files += glob.glob(os.path.join(base_folder, country, '*', 'labels.pkl'))
len(files)

In [ ]:
df_list = []
for file in files:
    tmp = pd.read_pickle(file)
    tmp = tmp[['geometry']]
    country = file.split('/')[-3]
    city = file.split('/')[-2]
    tmp['country'] = country
    tmp['city'] = city
    tmp = gpd.GeoDataFrame(tmp, geometry='geometry')
    tmp.to_crs(epsg=4326, inplace=True)
    df_list.append(tmp)
df = pd.concat(df_list, ignore_index=True)
df

In [ ]:
df =gpd.GeoDataFrame(df, geometry='geometry')
df['lat'] = df.geometry.centroid.y
df['lon'] = df.geometry.centroid.x
df = df.to_crs(epsg=3857)
df['area'] = df['geometry'].area / 1e6
df= df.to_crs(epsg=4326)
df

In [ ]:
world = gpd.read_file('../../data/processed/world.geojson')
world

In [ ]:
china_geo = world[(world['name'] == 'Taiwan') | (world['name'] == 'China')].unary_union
world.loc[31, 'geometry'] = china_geo
world = world[world['name'] != 'Taiwan']
world

In [ ]:
countries = {
    'AUS': 6,
    'BRA': 5,
    'CHN': 5,
    'FRA': 4,
    'NGA': 3,
    'PRT': 4,
    'USA': 4
}
selected_countries = world[world['id'].isin(countries.keys())]
selected_countries = selected_countries.merge(pd.DataFrame(countries, index=['#SDG']).T.reset_index().rename(columns={'index': 'id'}), on='id')
selected_countries

In [ ]:
proj = ccrs.Mercator()
selected_countries = selected_countries.to_crs(proj.proj4_init)

In [ ]:
selected_countries

In [ ]:
fig, ax = plt.subplots(figsize=(18/2.5, 8/2.5), 
                    #    dpi=300, 
                       subplot_kw={'projection':proj})

gl = ax.gridlines(crs=ccrs.PlateCarree(), draw_labels=True,
                  linewidth=1, color='gray', alpha=0.2, linestyle='--', xlocs=np.arange(-180, 180, 30), ylocs=np.arange(-90, 90, 30))

gl.top_labels = True    # show longitude labels on top
gl.bottom_labels = True # show longitude labels on bottom
gl.left_labels = True   # show latitude labels on left
gl.right_labels = True  # show latitude labels on right
gl.xlabel_style = {'size': 7, 'color': 'black'}  # longitude label size
gl.ylabel_style = {'size': 7, 'color': 'black'}  # latitude label size

ax.set_extent([-180, 180, -60, 60], crs=ccrs.PlateCarree())
ax.spines['left'].set_linewidth(1)
ax.spines['bottom'].set_linewidth(1)
ax.spines['right'].set_linewidth(1)
ax.spines['top'].set_linewidth(1)

# background map
ax.add_feature(cfeature.LAND, facecolor='#f0f0f0', edgecolor='none', zorder=0)
ax.add_feature(cfeature.BORDERS, linestyle=':', linewidth=0.3, zorder=1)
ax.coastlines(resolution='10m', linewidth=0.5, color='gray')

selected_countries.plot(
    column='#SDG',
    categorical=True,
    edgecolor='white',
    linewidth=0.1,
    zorder=10,
    cmap="summer_r",
    legend=True,
    legend_kwds={
        'fontsize':8,
        'title_fontsize': 8,
        'frameon': False,
        'title': 'Number of\nSDG\ncategories',
        'loc': 'lower left'
    },
    ax=ax
)
plt.savefig('../../data/figure_assets/numberSDG.png', bbox_inches='tight', dpi=300)
plt.show()

In [ ]:
fig, axes = plt.subplots(
    1, 4, figsize=(18/2.5, 10/2.5),
    subplot_kw={'projection': proj},
    gridspec_kw={'width_ratios': [5, 0.8, 2, 2.5]}
)

country_offset = {
    'US': [0.5, 20, 5],
    'Portugal': [0.5, 10, 5],
    'France': [0.5, 10, 5],
    'China': [0.01, 1, 0.5]
}

current_df = df[df['country'].isin(country_offset.keys())]

global_min, global_max = current_df['area'].min(), current_df['area'].max()

for ax, country in zip(axes, country_offset.keys()):
    tmp_df = df[df['country'] == country]
    offset, grid_offset_x, grid_offset_y = country_offset[country]
    max_x = tmp_df['lon'].max() + offset
    min_x = tmp_df['lon'].min() - offset
    max_y = tmp_df['lat'].max() + offset
    min_y = tmp_df['lat'].min() - offset

    ax.set_extent([min_x, max_x, min_y, max_y], crs=ccrs.PlateCarree())

    ax.add_feature(cfeature.LAND, facecolor='#f0f0f0', edgecolor='none', zorder=0)
    ax.add_feature(cfeature.BORDERS, linestyle=':', linewidth=0.3, zorder=1)
    ax.coastlines(resolution='10m', linewidth=0.5, color='gray')

    sns.scatterplot(
        data=tmp_df, x='lon', y='lat', hue='area',
        s=20, ax=ax, transform=ccrs.Geodetic(),
        hue_norm=(global_min, global_max),
        zorder=10, legend=False, palette='viridis', alpha=0.4
    )
    
# Colorbar uses the same cmap and norm.
norm = plt.Normalize(global_min, global_max)
cmap = plt.cm.get_cmap('viridis')
sm = plt.cm.ScalarMappable(cmap=cmap, norm=norm)
sm.set_array([])

# Add colorbar above the subplot.
cbar = fig.colorbar(
    sm, ax=axes, orientation='horizontal',
    location='top',       # place on top
    fraction=0.03,        # colorbar height fraction
    pad=0.03,             # padding from the subplot
    aspect=40             # aspect ratio (higher -> thinner)
)
# cbar.set_label('Area', labelpad=10)  # optional label
cbar.set_label('Area of urban neighborhoods')

# plt.tight_layout()
plt.savefig('../../data/figure_assets/top_spatial_units.png', bbox_inches='tight', dpi=300)
plt.show()

In [ ]:
fig, axes = plt.subplots(
    1, 3, figsize=(18/2.5, 10/2.5),
    subplot_kw={'projection': proj},
    gridspec_kw={'width_ratios': [1.25, 2, 4.75]}
)

country_offset = {
    'Brazil': [0.5, 20, 5],
    'Nigeria': [0.5, 10, 5],
    'Australia': [0.5, 10, 5],
}

current_df = df[df['country'].isin(country_offset.keys())]

global_min, global_max = current_df['area'].min(), current_df['area'].max()

for ax, country in zip(axes, country_offset.keys()):
    tmp_df = df[df['country'] == country]
    offset, grid_offset_x, grid_offset_y = country_offset[country]
    max_x = tmp_df['lon'].max() + offset
    min_x = tmp_df['lon'].min() - offset
    max_y = tmp_df['lat'].max() + offset
    min_y = tmp_df['lat'].min() - offset

    ax.set_extent([min_x, max_x, min_y, max_y], crs=ccrs.PlateCarree())

    ax.add_feature(cfeature.LAND, facecolor='#f0f0f0', edgecolor='none', zorder=0)
    ax.add_feature(cfeature.BORDERS, linestyle=':', linewidth=0.3, zorder=1)
    ax.coastlines(resolution='10m', linewidth=0.5, color='gray')

    sns.scatterplot(
        data=tmp_df, x='lon', y='lat', hue='area',
        s=20, ax=ax, transform=ccrs.Geodetic(),
        hue_norm=(global_min, global_max),
        zorder=10, legend=False, palette='viridis', alpha=0.4
    )
    
# Colorbar uses the same cmap and norm.
norm = plt.Normalize(global_min, global_max)
cmap = plt.cm.get_cmap('viridis')
sm = plt.cm.ScalarMappable(cmap=cmap, norm=norm)
sm.set_array([])

# Add colorbar above the subplot.
cbar = fig.colorbar(
    sm, ax=axes, orientation='horizontal',
    # location='top',       # place on top
    fraction=0.03,        # colorbar height fraction
    pad=0.03,             # padding from the subplot
    aspect=40             # aspect ratio (higher -> thinner)
)
# cbar.set_label('Area', labelpad=10)  # optional label
cbar.set_label('Area of urban neighborhoods')

# plt.tight_layout()
plt.savefig('../../data/figure_assets/bottom_spatial_units.png', bbox_inches='tight', dpi=300)
plt.show()